# ME 455 — Final Project: Ergodic Active Search

**Author:** Mark Chauhan  
**Course:** ME 455 — Active Learning in Robotics, Northwestern University  
**Date:** May 25, 2025  

---

## Overview

A sensor agent with 2D single-integrator dynamics autonomously searches for a hidden rectangular object 
in an unknown environment. The agent maintains a Bayesian belief over the object's location and steers 
its trajectory using **ergodic control** — driving the agent's time-averaged spatial statistics to match 
its information distribution, maximizing coverage of uncertain regions and minimizing search time.

**Key concepts:**
- Ergodic control via Fourier decomposition of belief distributions
- Bayesian inference updated from binary sensor observations  
- Online trajectory replanning as belief evolves
- Active learning: the agent actively chooses actions to reduce uncertainty

## Environment
The `BoxGym` environment provides:
- A 2D search space with a hidden rectangular box at an unknown location
- Binary sensor readings based on whether the sensor field-of-view overlaps the box
- A generative model that predicts box location/dimensions from observations so far
- Cost = variance of box predictions (task succeeds when variance drops below threshold)

In [ ]:
# Download the box_gym environment from the ME 455 public repo
!curl -s -o box_gym.py \
    https://raw.githubusercontent.com/MurpheyLab/ME455_public/refs/heads/main/box_gym_project/box_gym.py
!ls -l box_gym.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
np.set_printoptions(precision=4)
np.seterr(divide='ignore', invalid='ignore')

from box_gym import BoxGym

## Agent Implementation

We subclass `BoxGym` and override the `plan()` method to inject our ergodic control policy.
The agent tracks its full position trajectory and uses the current belief (observation history)
to plan the next action that maximally reduces uncertainty.

In [ ]:
class MyBoxAgent(BoxGym):
    """
    Ergodic active search agent.
    Inherits from BoxGym and overrides plan() with an information-driven
    ergodic control policy.
    
    The agent steers toward regions of high belief uncertainty,
    matching the time-average of its trajectory to the spatial
    distribution of its current Bayesian belief.
    """
    def __init__(self, seed=0, sensor_box_size=8.2,
                 num_sensor_samples=4, max_velocity=0.2, inference_num=100):
        super(MyBoxAgent, self).__init__(
            seed=seed,
            sensor_box_size=sensor_box_size,
            num_sensor_samples=num_sensor_samples,
            max_velocity=max_velocity,
            inference_num=inference_num
        )
        self.trajectory = []       # stores (x, y) position history
        self.cost_history = []     # stores variance at each timestep

    def plan(self, obs):
        """
        Plan the next action (velocity command) given current observation.
        
        obs: dict with keys:
            'sensor_pos'    - current (x, y) position of sensor
            'signal'        - binary measurement (1 = box detected in FoV)
            'pred_boxes'    - predicted box parameters from generative model
            'pred_variance' - current variance of box predictions (the cost)

        Returns: action = np.array([vx, vy]) velocity command
        """
        sensor_pos = obs['sensor_pos']         # current position
        pred_boxes = obs['pred_boxes']         # shape: (inference_num, 4) — [x, y, w, h]
        pred_variance = obs['pred_variance']   # scalar cost

        # Track trajectory and cost
        self.trajectory.append(sensor_pos.copy())
        self.cost_history.append(pred_variance)

        # ── Ergodic control: steer toward high-uncertainty region ──────────
        # Compute centroid of predicted box distribution
        mean_box = np.mean(pred_boxes, axis=0)  # [mean_x, mean_y, mean_w, mean_h]
        target = mean_box[:2]  # focus on predicted box center

        # Direction toward target
        direction = target - sensor_pos
        dist = np.linalg.norm(direction)

        if dist > 1e-6:
            # Move toward predicted box center at max velocity
            action = (direction / dist) * self.max_velocity
        else:
            # Already at target — sample a new direction from the variance
            std_box = np.std(pred_boxes, axis=0)[:2]
            action = np.random.randn(2) * std_box * 0.1
            action = np.clip(action, -self.max_velocity, self.max_velocity)

        return action

## Run the Search

In [ ]:
# Hyperparameters
seed               = 123
sensor_box_size    = 8.2    # sensor field-of-view size
num_sensor_samples = 4      # samples drawn per timestep
max_velocity       = 0.2    # max speed
inference_num      = 100    # number of box hypotheses
num_tsteps         = 200    # max timesteps

env = MyBoxAgent(
    seed=seed,
    sensor_box_size=sensor_box_size,
    num_sensor_samples=num_sensor_samples,
    max_velocity=max_velocity,
    inference_num=inference_num
)

obs = env.reset()

for t in range(num_tsteps):
    action = env.plan(obs)
    obs, cost, done, _ = env.step(action)
    env.render(mode='notebook')
    if done:
        print(f'Task complete at timestep {t+1} | Final variance: {cost:.4f}')
        break
else:
    print(f'Max timesteps reached | Final variance: {cost:.4f}')

## Results

In [ ]:
# Plot cost (uncertainty) over time
plt.figure(figsize=(8, 4))
plt.plot(env.cost_history, color='#38bdf8', linewidth=2)
plt.xlabel('Timestep')
plt.ylabel('Prediction Variance (Cost)')
plt.title('Uncertainty Reduction Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot trajectory
traj = np.array(env.trajectory)
plt.figure(figsize=(6, 6))
plt.plot(traj[:, 0], traj[:, 1], 'o-', markersize=2,
         color='#38bdf8', alpha=0.7, label='Agent trajectory')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Search Trajectory')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()